In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
import tensorflow as tf
from keras.layers import BatchNormalization
from keras.utils import to_categorical  # Updated import
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split

In [2]:
def create_dataframe(data_dir):
    data = [
        {"image_path": os.path.join(data_dir, dir_name, fname), "label": label}
        for label, dir_name in enumerate(os.listdir(data_dir))
        for fname in os.listdir(os.path.join(data_dir, dir_name))
    ]
    return pd.DataFrame(data)

train_dir = 'dataset/Train'
test_dir = 'dataset/Test'

# Create combined dataframe
df = pd.concat([create_dataframe(train_dir), create_dataframe(test_dir)], ignore_index=True)

In [3]:
df.sample(10)

,image_path,label
553,dataset/Test\squamous cell carcinoma\ISIC_0012...,7
478,dataset/Test\basal cell carcinoma\ISIC_0024454...,1
484,dataset/Test\dermatofibroma\ISIC_0011410.jpg,2
32,dataset/Train\actinic keratosis\ISIC_0027562.jpg,0
163,dataset/Train\melanoma\ISIC_0000153.jpg,3
93,dataset/Train\basal cell carcinoma\ISIC_002551...,1
488,dataset/Test\dermatofibroma\ISIC_0011865.jpg,2
171,dataset/Train\melanoma\ISIC_0000161.jpg,3
33,dataset/Train\actinic keratosis\ISIC_0027580.jpg,0
154,dataset/Train\melanoma\ISIC_0000144.jpg,3


In [4]:
# Get list of directories and create label_map
label_map = {i: label for i, label in enumerate(os.listdir(train_dir))}
num_classes = len(label_map)
label_map

{0: 'actinic keratosis',
 1: 'basal cell carcinoma',
 2: 'dermatofibroma',
 3: 'melanoma',
 4: 'nevus',
 5: 'pigmented benign keratosis',
 6: 'seborrheic keratosis',
 7: 'squamous cell carcinoma',
 8: 'vascular lesion'}

In [5]:
max_images_per_class = 300

# Limit each class to max_images_per_class samples
df = df.groupby("label").head(max_images_per_class).reset_index(drop=True)

In [6]:
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as e:
    print(e)

import multiprocessing

# Get the number of CPU cores
max_workers = multiprocessing.cpu_count()
print(max_workers)

[]
4


In [7]:
import concurrent.futures
# Function to resize image arrays
def resize_image_array(image_path):
    return np.asarray(Image.open(image_path).resize((100, 75)))

# Parallelize resizing process
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    df['image'] = list(executor.map(resize_image_array, df['image_path']))

In [8]:
# Initialize ImageDataGenerator with transformations
datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.5,
    height_shift_range=0.25,
    shear_range=0.25,
    zoom_range=0.25,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [9]:
# Initialize augmented DataFrame
augmented_df = pd.DataFrame(columns=['image_path', 'label', 'image'])

for class_label in df['label'].unique():
    # Get image arrays for the current class
    class_images = df[df['label'] == class_label]
    image_arrays = class_images['image'].values
    num_images_needed = max_images_per_class - len(image_arrays)
    
    # Add original images to augmented DataFrame
    augmented_df = pd.concat([augmented_df, class_images], ignore_index=True)

    if num_images_needed > 0:
        selected_images = np.random.choice(image_arrays, size=num_images_needed, replace=True)
        
        # Augment and add new images
        for img_array in selected_images:
            image_tensor = np.expand_dims(img_array, axis=0)
            augmented_images = datagen.flow(image_tensor, batch_size=1)
            
            for _ in range(1):  # Generate one augmented image at a time
                augmented_img = augmented_images.next()[0].astype('uint8')
                new_row = pd.DataFrame([{'image_path': None, 'label': class_label, 'image': augmented_img}])
                augmented_df = pd.concat([augmented_df, new_row], ignore_index=True)

# Limit the size of each class
df = augmented_df.groupby('label').head(max_images_per_class).sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
# Count the number of images in each class
class_counts = df['label'].value_counts().sort_index()

# Print the dataset summary
print("Dataset Summary")
print("-" * 60)
print(f"{'Class Label':<15} {'Class Name':<30} {'Count':<10}")
print("-" * 60)
for label, name in label_map.items():
    print(f"{label:<15} {name:<30} {class_counts[label]:<10}")
print("-" * 60)
print(f"{'Total':<45} {class_counts.sum():<10}")

Dataset Summary
------------------------------------------------------------
Class Label     Class Name                     Count     
------------------------------------------------------------
0               actinic keratosis              300       
1               basal cell carcinoma           300       
2               dermatofibroma                 300       
3               melanoma                       300       
4               nevus                          300       
5               pigmented benign keratosis     300       
6               seborrheic keratosis           300       
7               squamous cell carcinoma        300       
8               vascular lesion                300       
------------------------------------------------------------
Total                                         2700      


In [11]:
X = df.drop(columns=['label','image_path'],axis=1)
y = df['label']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,shuffle=True)

In [13]:
# Convert images to numpy arrays
X_train = np.asarray(X_train['image'].tolist())
X_test = np.asarray(X_test['image'].tolist())

# Normalize images
X_train_mean, X_train_std = X_train.mean(), X_train.std()
X_test_mean, X_test_std = X_test.mean(), X_test.std()

X_train = (X_train - X_train_mean) / X_train_std
X_test = (X_test - X_test_mean) / X_test_std

In [14]:
from keras.utils import to_categorical

# One-hot encode the labels
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

In [15]:
# Split data into training and validation sets
X_train, X_validate, y_train, y_validate = train_test_split(X_train, y_train, test_size=0.2, shuffle=True)

In [16]:
# Reshape images to 3D (height=75px, width=100px, channels=3)
X_train = X_train.reshape(X_train.shape[0], 75, 100, 3)
X_test = X_test.reshape(X_test.shape[0], 75, 100, 3)
X_validate = X_validate.reshape(X_validate.shape[0], 75, 100, 3)

In [17]:
y_train = y_train.astype(int)
y_validate = y_validate.astype(int)

In [18]:
# Count the number of images per class
train_counts = np.sum(y_train, axis=0)
val_counts = np.sum(y_validate, axis=0)
test_counts = np.sum(y_test, axis=0)

# Print dataset summary
print("Dataset Summary")
print("-" * 90)
print(f"{'Class Label':<15} {'Class Name':<30} {'Train':<10} {'Validation':<12} {'Test':<10} {'Total':<10}")
print("-" * 90)
for label, name in label_map.items():
    train_num = int(train_counts[label])
    val_num = int(val_counts[label])
    test_num = int(test_counts[label])
    total_num = train_num + val_num + test_num
    print(f"{label:<15} {name:<30} {train_num:<10} {val_num:<12} {test_num:<10} {total_num:<10}")
print("-" * 90)
total_images = len(y_train) + len(y_validate) + len(y_test)
print(f"{'Total':<46} {len(y_train):<10} {len(y_validate):<12} {len(y_test):<10} {total_images:<10}")

Dataset Summary
------------------------------------------------------------------------------------------
Class Label     Class Name                     Train      Validation   Test       Total     
------------------------------------------------------------------------------------------
0               actinic keratosis              203        40           57         300       
1               basal cell carcinoma           202        43           55         300       
2               dermatofibroma                 154        67           79         300       
3               melanoma                       204        43           53         300       
4               nevus                          193        48           59         300       
5               pigmented benign keratosis     202        54           44         300       
6               seborrheic keratosis           184        53           63         300       
7               squamous cell carcinoma        198        

In [19]:
input_shape = df['image'][0].shape
input_shape

(75, 100, 3)

In [20]:
from keras.applications import MobileNetV3Large

In [21]:
from keras.applications.resnet import preprocess_input as resnet_preprocess_input
from keras.applications import DenseNet201
from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import SGD

In [22]:
# Define the DenseNet201 model with custom layers
model = Sequential([
    DenseNet201(include_top=False, weights='imagenet', input_shape=input_shape),
    Flatten(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

74836368/74836368 [==============================] - 13s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 densenet201 (Functional)    (None, 2, 3, 1920)        18321984  
                                                                 
 flatten (Flatten)           (None, 11520)             0         
                                                                 
 dropout (Dropout)           (None, 11520)             0         
                                                                 
 dense (Dense)               (None, 512)               5898752   
                                                                 
 dense_1 (Dense)             (None, 9)                 4617      
                                                                 
Total params: 24,225,353
Trainable params: 23,996,297
Non-trainable params: 229,056
______________________________________

In [23]:
# Compile the model
opt = SGD(learning_rate=0.001, momentum=0.9)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Set learning rate annealer
learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=3,
    verbose=1,
    factor=0.5,
    min_lr=0.00001
)

In [24]:
%%time
# Fit the model
history = model.fit(
    X_train, y_train,
    epochs=50, batch_size=32,
    validation_data=(X_validate, y_validate),
    callbacks=[learning_rate_reduction]
)

Epoch 1/50
54/54 [==============================] - 491s 8s/step - loss: 2.0491 - accuracy: 0.3322 - val_loss: 3.9913 - val_accuracy: 0.3079 - lr: 0.0010
Epoch 2/50
54/54 [==============================] - 417s 8s/step - loss: 0.9948 - accuracy: 0.6661 - val_loss: 1.5895 - val_accuracy: 0.5069 - lr: 0.0010
Epoch 3/50
54/54 [==============================] - 413s 8s/step - loss: 0.5876 - accuracy: 0.7928 - val_loss: 1.1964 - val_accuracy: 0.6481 - lr: 0.0010
Epoch 4/50
54/54 [==============================] - 410s 8s/step - loss: 0.3083 - accuracy: 0.8947 - val_loss: 1.0391 - val_accuracy: 0.7222 - lr: 0.0010
Epoch 5/50
54/54 [==============================] - 415s 8s/step - loss: 0.1802 - accuracy: 0.9404 - val_loss: 0.9263 - val_accuracy: 0.7361 - lr: 0.0010
Epoch 6/50
54/54 [==============================] - 420s 8s/step - loss: 0.0903 - accuracy: 0.9728 - val_loss: 0.8587 - val_accuracy: 0.7593 - lr: 0.0010
Epoch 7/50
54/54 [==============================] - 413s 8s/step - loss: 0.0

In [25]:
model.save('skin_disease_model.h5')

In [ ]:
from keras.models import load_model
from keras.preprocessing import image
import numpy as np

In [27]:
model = load_model('skin_disease_model.h5')

In [29]:
# Define the path to the image
image_path='dataset/Test/nevus/ISIC_0000015.jpg'

# Load the image and preprocess it
img = image.load_img(image_path, target_size=(75, 100))  # Correct order: height=75px, width=100px
img_array = image.img_to_array(img)  # Convert image to array
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Normalize the image as per the training data preprocessing
img_array = (img_array - np.mean(img_array)) / np.std(img_array)

In [30]:
# Predict the class
predictions = model.predict(img_array)
predicted_class = np.argmax(predictions, axis=1)

# Map the predicted class index to the class label
predicted_label = label_map[predicted_class[0]]

print(f"Predicted Class: {predicted_label}")

1/1 [==============================] - 8s 8s/step
Predicted Class: nevus
